### Import the nessessory Liberaries

In [8]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from feature_engine.outliers import Winsorizer
import joblib

In [10]:
# Load dataset
dataset = pd.read_csv(r"CarswithEnginetype.xls")


In [14]:
dataset.head()

,MPG,Enginetype,HP,VOL,SP,WT
0,53.700681,petrol,49.0,89.0,104.185353,28.762059
1,50.013401,hybrid,55.0,92.0,105.461264,30.466833
2,50.013401,diesel,55.0,92.0,105.461264,30.193597
3,45.696322,lpg,70.0,92.0,113.461264,30.632114
4,50.504232,petrol,53.0,92.0,104.461264,29.889149


In [18]:
dataset.size

486

In [20]:
dataset.shape

(81, 6)

In [22]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81 entries, 0 to 80
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MPG         81 non-null     float64
 1   Enginetype  81 non-null     object 
 2   HP          80 non-null     float64
 3   VOL         80 non-null     float64
 4   SP          80 non-null     float64
 5   WT          80 non-null     float64
dtypes: float64(5), object(1)
memory usage: 3.9+ KB


In [26]:
dataset.describe()

,MPG,HP,VOL,SP,WT
count,81.000000,80.000000,80.000000,80.000000,80.000000
mean,34.422076,117.787500,98.850000,121.080767,32.346948
std,9.131445,57.401472,22.429129,13.650623,7.516623
min,12.101263,49.000000,50.000000,99.564907,15.712859
25%,27.856252,83.250000,89.000000,113.743011,29.577772
50%,35.152727,100.000000,101.000000,117.938624,32.718081
75%,39.531633,140.000000,113.000000,126.137156,37.288925
max,53.700681,322.000000,160.000000,169.598513,52.997752


In [28]:
dataset.corr(numeric_only=True)

,MPG,HP,VOL,SP,WT
MPG,1.000000,-0.724239,-0.532726,-0.677916,-0.524217
HP,-0.724239,1.000000,0.063045,0.972857,0.069146
VOL,-0.532726,0.063045,1.000000,0.083075,0.999205
SP,-0.677916,0.972857,0.083075,1.000000,0.089692
WT,-0.524217,0.069146,0.999205,0.089692,1.000000


In [30]:
# Separate features and target
X = dataset.iloc[:, 1:]  # Features
y = dataset.iloc[:, 0]   # Target


In [34]:
X.head()

,Enginetype,HP,VOL,SP,WT
0,petrol,49.0,89.0,104.185353,28.762059
1,hybrid,55.0,92.0,105.461264,30.466833
2,diesel,55.0,92.0,105.461264,30.193597
3,lpg,70.0,92.0,113.461264,30.632114
4,petrol,53.0,92.0,104.461264,29.889149


In [36]:
y.head()

0    53.700681
1    50.013401
2    50.013401
3    45.696322
4    50.504232
Name: MPG, dtype: float64

In [38]:
# Identify categorical and numerical columns
numeric_features = X.select_dtypes(exclude=['object']).columns
categorical_features = X.select_dtypes(include=['object']).columns


In [40]:
numeric_features

Index(['HP', 'VOL', 'SP', 'WT'], dtype='object')

In [42]:
categorical_features

Index(['Enginetype'], dtype='object')

In [46]:
# Numerical pipeline
num_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='mean')),
    ('winsorize', Winsorizer(capping_method='iqr', tail='both', fold=1.5)),
    ('scale', MinMaxScaler())
])

# Categorical pipeline
categ_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder( drop='first'))
])

In [48]:
# ColumnTransformer for all features
preprocess_pipeline = ColumnTransformer([
    ('numerical', num_pipeline, numeric_features),
    ('categorical', categ_pipeline, categorical_features)
])

In [50]:
# Fit the pipeline on X (features only)
clean = preprocess_pipeline.fit(X)

In [52]:
clean

ColumnTransformer(transformers=[('numerical',
                                 Pipeline(steps=[('impute', SimpleImputer()),
                                                 ('winsorize',
                                                  Winsorizer(capping_method='iqr',
                                                             fold=1.5,
                                                             tail='both')),
                                                 ('scale', MinMaxScaler())]),
                                 Index(['HP', 'VOL', 'SP', 'WT'], dtype='object')),
                                ('categorical',
                                 Pipeline(steps=[('impute',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encode',
                                                  OneHotEncoder(drop='first'))]),
                                 Index(['Enginetype'], dtype='object'))])

In [54]:
# Transform features and convert to DataFrame
Clean_data = pd.DataFrame(clean.transform(X))#, columns=clean.get_feature_names_out())

In [56]:
Clean_data.head()

,0,1,2,3,4,5,6,7
0,0.000000,0.37500,0.103108,0.347930,0.0,0.0,0.0,1.0
1,0.034286,0.40625,0.131581,0.403550,0.0,1.0,0.0,0.0
2,0.034286,0.40625,0.131581,0.394635,1.0,0.0,0.0,0.0
3,0.120000,0.40625,0.310106,0.408942,0.0,0.0,1.0,0.0
4,0.022857,0.40625,0.109265,0.384702,0.0,0.0,0.0,1.0


In [58]:
import os
os.getcwd()

'C:\\Users\\Om Computers\\Desktop\\Certified Data Science\\03-Machine Learning Associate\\Linear regression'

In [60]:
# Save the preprocessing pipeline
joblib.dump(clean, 'preprocess_pipeline.pkl')


['preprocess_pipeline.pkl']

In [46]:

# Check the resulting DataFrame
print(Clean_data.head(10))


          0         1         2         3    4    5    6    7
0  0.000000  0.375000  0.103108  0.347930  0.0  0.0  0.0  1.0
1  0.034286  0.406250  0.131581  0.403550  0.0  1.0  0.0  0.0
2  0.034286  0.406250  0.131581  0.394635  1.0  0.0  0.0  0.0
3  0.120000  0.406250  0.310106  0.408942  0.0  0.0  1.0  0.0
4  0.022857  0.406250  0.109265  0.384702  0.0  0.0  0.0  1.0
5  0.120000  0.375000  0.303949  0.375000  0.0  0.0  0.0  1.0
6  0.034286  0.500000  0.131581  0.398383  0.0  0.0  0.0  0.0
7  0.074286  0.000000  0.067697  0.000000  0.0  0.0  0.0  0.0
8  0.074286  0.000000  0.067697  0.000000  0.0  0.0  1.0  0.0
9  0.177143  0.427083  0.358842  0.418340  0.0  0.0  0.0  1.0


In [63]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

In [65]:
# X_encoded = preprocessed features
# y = target variable
X_train, X_test, y_train, y_test = train_test_split(
    Clean_data, y, test_size=0.3, random_state=14
)


In [67]:
X_train.shape

(56, 8)

In [71]:
X_test.shape

(25, 8)

In [73]:
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [75]:
y_pred = lr.predict(X_test)

In [79]:
y_pred.shape

(25,)

In [81]:
y_test.shape

(25,)

In [45]:
y_pred

array([34.97066764, 43.8488357 , 36.61448337, 37.17209652, 40.38731707,
       40.13682801, 40.93334903, 29.0869202 , 42.71582082, 38.18139825,
       34.48734237, 33.51890745, 23.01628698, 38.55998251, 22.69604554,
       37.06820316, 22.69045627])

In [85]:
y_test

29    35.404192
35    37.958743
44    35.643558
6     50.013401
2     50.013401
79    19.833733
55    27.856252
33    36.285456
58    30.131923
49    29.629936
73    19.086341
24    38.310606
8     46.716554
23    40.474723
39    31.014131
80    12.101263
36    37.958743
34    39.531633
48    31.014131
60    27.354265
42    34.070668
52    29.629936
20    38.411003
67    23.103172
16    42.789909
Name: MPG, dtype: float64

In [83]:
# R-squared
r2 = r2_score(y_test, y_pred)
print("R-squared:", r2)

# Mean Squared Error
mse = mean_squared_error(y_test, y_pred)
print("Mean Squared Error:", mse)

# Optional: Root Mean Squared Error
rmse = mse ** 0.5
print("RMSE:", rmse)

R-squared: 0.8043200203551253
Mean Squared Error: 16.25433618403731
RMSE: 4.0316666757108415


In [39]:
import joblib

# Save preprocessing pipeline
joblib.dump(clean, 'preprocessing_pipeline.pkl')  # Saves in current working directory
preprocess_pipeline = joblib.load('preprocessing_pipeline.pkl')

In [89]:
# Assuming mlr is your trained LinearRegression model
joblib.dump(lr, 'mlr_model.pkl')  # Saves in current working directory
lr_model = joblib.load('mlr_model.pkl')

In [1]:
27.4**0.5

5.2345009313209605